# Training Models (iBVPNet & FactorizePhys) for Group F

This notebook contains the training pipeline for iBVPNet and FactorizePhys using the preprocessed Group F dataset.
import csv
import time
import math
import glob


In [2]:
import os
import sys
import glob
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Setup REPO_ROOT
REPO_ROOT = "/home/iec/MinhHieu/Non-Invasive/rPPG"


## Inlined source

The cells below contain the model / loss source that was previously imported from the `neural_methods/` (and `evaluation/`) packages. They are inlined here so the notebook is self-contained.


In [ ]:
# === inlined from neural_methods/model/iBVPNet.py ===
"""iBVPNet - 3D Convolutional Network.
Proposed along with the iBVP Dataset, see https://doi.org/10.3390/electronics13071334

Joshi, Jitesh, and Youngjun Cho. 2024. "iBVP Dataset: RGB-Thermal rPPG Dataset with High Resolution Signal Quality Labels" Electronics 13, no. 7: 1334.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class ConvBlock3D(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding):
        super(ConvBlock3D, self).__init__()
        self.conv_block_3d = nn.Sequential(
            nn.Conv3d(in_channel, out_channel, kernel_size, stride, padding),
            nn.Tanh(),
            nn.InstanceNorm3d(out_channel),
        )

    def forward(self, x):
        return self.conv_block_3d(x)


class DeConvBlock3D(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding):
        super(DeConvBlock3D, self).__init__()
        k_t, k_s1, k_s2 = kernel_size
        s_t, s_s1, s_s2 = stride
        self.deconv_block_3d = nn.Sequential(
            nn.ConvTranspose3d(in_channel, in_channel, (k_t, 1, 1), (s_t, 1, 1), padding),
            nn.Tanh(),
            nn.InstanceNorm3d(in_channel),
            
            nn.Conv3d(in_channel, out_channel, (1, k_s1, k_s2), (1, s_s1, s_s2), padding),
            nn.Tanh(),
            nn.InstanceNorm3d(out_channel),
        )

    def forward(self, x):
        return self.deconv_block_3d(x)

# num_filters
nf = [8, 16, 24, 40, 64]

class encoder_block(nn.Module):
    def __init__(self, in_channel, debug=False):
        super(encoder_block, self).__init__()
        # in_channel, out_channel, kernel_size, stride, padding

        self.debug = debug
        self.spatio_temporal_encoder = nn.Sequential(
            ConvBlock3D(in_channel, nf[0], [1, 3, 3], [1, 1, 1], [0, 1, 1]),
            ConvBlock3D(nf[0], nf[1], [3, 3, 3], [1, 1, 1], [1, 1, 1]),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2)),
            ConvBlock3D(nf[1], nf[2], [1, 3, 3], [1, 1, 1], [0, 1, 1]),
            ConvBlock3D(nf[2], nf[3], [3, 3, 3], [1, 1, 1], [1, 1, 1]),
            nn.MaxPool3d((1, 2, 2), stride=(1, 2, 2)),
            ConvBlock3D(nf[3], nf[4], [1, 3, 3], [1, 1, 1], [0, 1, 1]),
            ConvBlock3D(nf[4], nf[4], [3, 3, 3], [1, 1, 1], [1, 1, 1]),
        )

        self.temporal_encoder = nn.Sequential(
            ConvBlock3D(nf[4], nf[4], [11, 1, 1], [1, 1, 1], [5, 0, 0]),
            ConvBlock3D(nf[4], nf[4], [11, 3, 3], [1, 1, 1], [5, 1, 1]),
            nn.MaxPool3d((2, 2, 2), stride=(2, 2, 2)),
            ConvBlock3D(nf[4], nf[4], [11, 1, 1], [1, 1, 1], [5, 0, 0]),
            ConvBlock3D(nf[4], nf[4], [11, 3, 3], [1, 1, 1], [5, 1, 1]),
            nn.MaxPool3d((2, 2, 2), stride=(2, 1, 1)),
            ConvBlock3D(nf[4], nf[4], [7, 1, 1], [1, 1, 1], [3, 0, 0]),
            ConvBlock3D(nf[4], nf[4], [7, 3, 3], [1, 1, 1], [3, 1, 1])
        )

    def forward(self, x):
        if self.debug:
            print("Encoder")
            print("x.shape", x.shape)
        st_x = self.spatio_temporal_encoder(x)
        if self.debug:
            print("st_x.shape", st_x.shape)
        t_x = self.temporal_encoder(st_x)
        if self.debug:
            print("t_x.shape", t_x.shape)
        return t_x


class decoder_block(nn.Module):
    def __init__(self, debug=False):
        super(decoder_block, self).__init__()
        self.debug = debug
        self.decoder_block = nn.Sequential(
            DeConvBlock3D(nf[4], nf[3], [7, 3, 3], [2, 2, 2], [2, 1, 1]),
            DeConvBlock3D(nf[3], nf[2], [7, 3, 3], [2, 2, 2], [2, 1, 1])
        )

    def forward(self, x):
        if self.debug:
            print("Decoder")
            print("x.shape", x.shape)
        x = self.decoder_block(x)
        if self.debug:
            print("x.shape", x.shape)
        return x



class iBVPNet(nn.Module):
    def __init__(self, frames, in_channels=3, debug=False):
        super(iBVPNet, self).__init__()
        self.debug = debug

        self.in_channels = in_channels
        if self.in_channels == 1 or self.in_channels == 3:
            self.norm = nn.InstanceNorm3d(self.in_channels)
        elif self.in_channels == 4:
            self.rgb_norm = nn.InstanceNorm3d(3)
            self.thermal_norm = nn.InstanceNorm3d(1)
        else:
            print("Unsupported input channels")

        self.ibvpnet = nn.Sequential(
            encoder_block(in_channels, debug),
            decoder_block(debug),
            # spatial adaptive pooling
            nn.AdaptiveMaxPool3d((frames, 1, 1)),
            nn.Conv3d(nf[2], 1, [1, 1, 1], stride=1, padding=0)
        )

        
    def forward(self, x): # [batch, Features=3, Temp=frames, Width=32, Height=32]
        
        [batch, channel, length, width, height] = x.shape

        x = torch.diff(x, dim=2)

        if self.debug:
            print("Input.shape", x.shape)

        if self.in_channels == 1:
            x = self.norm(x[:, -1:, :, :, :])
        elif self.in_channels == 3:
            x = self.norm(x[:, :3, :, :, :])
        elif self.in_channels == 4:
            rgb_x = self.rgb_norm(x[:, :3, :, :, :])
            thermal_x = self.thermal_norm(x[:, -1:, :, :, :])
            x = torch.concat([rgb_x, thermal_x], dim = 1)
        else:
            try:
                print("Specified input channels:", self.in_channels)
                print("Data channels", channel)
                assert self.in_channels <= channel
            except:
                print("Incorrectly preprocessed data provided as input. Number of channels exceed the specified or default channels")
                print("Default or specified channels:", self.in_channels)
                print("Data channels [B, C, N, W, H]", x.shape)
                print("Exiting")
                exit()

        if self.debug:
            print("Diff Normalized shape", x.shape)

        feats = self.ibvpnet(x)
        if self.debug:
            print("feats.shape", feats.shape)
        rPPG = feats.view(-1, length-1)
        return rPPG
    

if __name__ == "__main__":
    import torch
    from torch.utils.tensorboard import SummaryWriter

    # default `log_dir` is "runs" - we'll be more specific here
    writer = SummaryWriter('runs/iBVPNet')

    duration = 8
    fs = 25
    batch_size = 4
    frames = duration*fs
    in_channels = 1
    height = 64
    width = 64
    test_data = torch.rand(batch_size, in_channels, frames, height, width)

    net = iBVPNet(in_channels=in_channels, frames=frames, debug=True)
    # print("-"*100)
    # print(net)
    # print("-"*100)
    pred = net(test_data)

    print(pred.shape)

    writer.add_graph(net, test_data)
    writer.close()


In [ ]:
# === inlined from neural_methods/model/FactorizePhys/FSAM.py ===
"""
FactorizePhys: Matrix Factorization for Multidimensional Attention in Remote Physiological Sensing
NeurIPS 2024
Jitesh Joshi, Sos S. Agaian, and Youngjun Cho
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.modules.batchnorm import _BatchNorm
import numpy as np
import neurokit2 as nk


class _MatrixDecompositionBase(nn.Module):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__()

        self.dim = dim
        self.md_type = md_config["MD_TYPE"]
        if dim == "3D":
            self.transform = md_config["MD_TRANSFORM"]
        self.S = md_config["MD_S"]
        self.R = md_config["MD_R"]
        self.debug = debug

        self.train_steps = md_config["MD_STEPS"]
        self.eval_steps = md_config["MD_STEPS"]

        self.inv_t = md_config["INV_T"]
        self.eta = md_config["ETA"]

        self.rand_init = md_config["RAND_INIT"]
        self.device = device

        # print('Dimension:', self.dim)
        # print('S', self.S)
        # print('D', self.D)
        # print('R', self.R)
        # print('train_steps', self.train_steps)
        # print('eval_steps', self.eval_steps)
        # print('inv_t', self.inv_t)
        # print('eta', self.eta)
        # print('rand_init', self.rand_init)

    def _build_bases(self, B, S, D, R):
        raise NotImplementedError

    def local_step(self, x, bases, coef):
        raise NotImplementedError

    @torch.no_grad()
    def local_inference(self, x, bases):
        # (B * S, D, N)^T @ (B * S, D, R) -> (B * S, N, R)
        coef = torch.bmm(x.transpose(1, 2), bases)
        coef = F.softmax(self.inv_t * coef, dim=-1)

        steps = self.train_steps if self.training else self.eval_steps
        for _ in range(steps):
            bases, coef = self.local_step(x, bases, coef)

        return bases, coef

    def compute_coef(self, x, bases, coef):
        raise NotImplementedError

    def forward(self, x, return_bases=False):

        if self.debug:
            print("Org x.shape", x.shape)

        if self.dim == "3D":        # (B, C, T, H, W) -> (B * S, D, N)
            B, C, T, H, W = x.shape

            # t = Time, k = Channels, a & B = height and width
            if self.transform.lower() == "t_kab":
                # # dimension of vector of our interest is T (rPPG signal as T dimension), so forming this as vector
                # # From spatial and channel dimension, which are features, only 2-4 shall be enough to generate the approximated attention matrix
                D = T // self.S
                N = C * H * W

            elif self.transform.lower() == "tk_ab":
                D = T * C // self.S
                N = H * W

            elif self.transform.lower() == "k_tab":
                D = C // self.S
                N = T * H * W

            else:
                print("Invalid MD_TRANSFORM specified:", self.transform)
                exit()

            # # smoothening the temporal dimension
            # x = x.view(B * self.S, N, D)
            # # print("Intermediate-1 x", x.shape)

            # sample_1 = x[:, :, 0].unsqueeze(2)
            # sample_2 = x[:, :, -1].unsqueeze(2)
            # x = torch.cat([sample_1, x, sample_2], dim=2)
            # gaussian_kernel = [1.0, 1.0, 1.0]
            # kernels = torch.FloatTensor([[gaussian_kernel]]).repeat(N, N, 1).to(self.device)
            # bias = torch.FloatTensor(torch.zeros(N)).to(self.device)
            # x = F.conv1d(x, kernels, bias=bias, padding="valid")
            # x = (x - x.min()) / (x.max() - x.min())

            # x = x.permute(0, 2, 1)
            # # print("Intermediate-2 x", x.shape)

            x = x.view(B * self.S, D, N)

        elif self.dim == "2D":      # (B, C, H, W) -> (B * S, D, N)
            B, C, H, W = x.shape
            D = C // self.S
            N = H * W
            x = x.view(B * self.S, D, N)

        elif self.dim == "2D_TSM":  # (B*frame_depth, C, H, W) -> (B, D, N)
            B, C, H, W = x.shape
            BN = B
            B = B // self.S
            D = self.S
            N = C * H * W
            x = x.view(B, D, N)
            self.S = 1  # re-setting this for local inference

        elif self.dim == "1D":                       # (B, C, L) -> (B * S, D, N)
            B, C, L = x.shape
            D = L // self.S
            N = C
            x = x.view(B * self.S, D, N)

        else:
            print("Dimension not supported")
            exit()

        if self.debug:
            print("MD_Type", self.md_type)
            print("MD_S", self.S)
            print("MD_D", D)
            print("MD_N", N)
            print("MD_R", self.R)
            print("MD_TRAIN_STEPS", self.train_steps)
            print("MD_EVAL_STEPS", self.eval_steps)
            print("x.view(B * self.S, D, N)", x.shape)

        if not self.rand_init and not hasattr(self, 'bases'):
            bases = self._build_bases(1, self.S, D, self.R)
            self.register_buffer('bases', bases)

        # (S, D, R) -> (B * S, D, R)
        if self.rand_init:
            bases = self._build_bases(B, self.S, D, self.R)
        else:
            bases = self.bases.repeat(B, 1, 1).to(self.device)

        bases, coef = self.local_inference(x, bases)

        # (B * S, N, R)
        coef = self.compute_coef(x, bases, coef)

        # (B * S, D, R) @ (B * S, N, R)^T -> (B * S, D, N)
        x = torch.bmm(bases, coef.transpose(1, 2))


        if self.dim == "3D":

            apply_smoothening = False
            if apply_smoothening:
                # smoothening the temporal dimension
                x = x.view(B, D * self.S, N)    #Joining temporal dimension for contiguous smoothening
                # print("Intermediate-0 x", x.shape)            
                x = x.permute(0, 2, 1)
                # print("Intermediate-1 x", x.shape)

                sample_1 = x[:, :, 0].unsqueeze(2)
                # sample_2 = x[:, :, 0].unsqueeze(2)
                sample_3 = x[:, :, -1].unsqueeze(2)
                # sample_4 = x[:, :, -1].unsqueeze(2)
                x = torch.cat([sample_1, x, sample_3], dim=2)
                # x = torch.cat([sample_1, sample_2, x, sample_3, sample_4], dim=2)
                # gaussian_kernel = [0.25, 0.50, 0.75, 0.50, 0.25]
                # gaussian_kernel = [0.33, 0.66, 1.00, 0.66, 0.33]
                # gaussian_kernel = [0.3, 0.7, 1.0, 0.7, 0.3]
                # gaussian_kernel = [0.3, 1.0, 1.0, 1.0, 0.3]
                # gaussian_kernel = [0.20, 0.80, 1.00, 0.80, 0.20]
                # gaussian_kernel = [1.0, 1.0, 1.0]
                gaussian_kernel = [0.8, 1.0, 0.8]
                kernels = torch.FloatTensor([[gaussian_kernel]]).repeat(N, N, 1).to(self.device)
                bias = torch.FloatTensor(torch.zeros(N)).to(self.device)
                x = F.conv1d(x, kernels, bias=bias, padding="valid")
                # x = (x - x.min()) / (x.max() - x.min())
                # x = (x - x.mean()) / (x.std())
                # x = x - x.min()
                x = (x - x.min())/(x.std())

                # print("Intermediate-2 x", x.shape)

            # (B * S, D, N) -> (B, C, T, H, W)
            x = x.view(B, C, T, H, W)
        elif self.dim == "2D":
            # (B * S, D, N) -> (B, C, H, W)
            x = x.view(B, C, H, W)

        elif self.dim == "2D_TSM":
            # (B, D, N) -> (B, C, H, W)
            x = x.view(BN, C, H, W)

        else:
            # (B * S, D, N) -> (B, C, L)
            x = x.view(B, C, L)

        # (B * L, D, R) -> (B, L, N, D)
        bases = bases.view(B, self.S, D, self.R)

        if not self.rand_init and not self.training and not return_bases:
            self.online_update(bases)

        # if not self.rand_init or return_bases:
        #     return x, bases
        # else:
        return x

    @torch.no_grad()
    def online_update(self, bases):
        # (B, S, D, R) -> (S, D, R)
        update = bases.mean(dim=0)
        self.bases += self.eta * (update - self.bases)
        self.bases = F.normalize(self.bases, dim=1)


class NMF(_MatrixDecompositionBase):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__(device, md_config, debug=debug, dim=dim)
        self.device = device
        self.inv_t = 1

    def _build_bases(self, B, S, D, R):
        # bases = torch.rand((B * S, D, R)).to(self.device)
        bases = torch.ones((B * S, D, R)).to(self.device)
        bases = F.normalize(bases, dim=1)

        return bases

    @torch.no_grad()
    def local_step(self, x, bases, coef):
        # (B * S, D, N)^T @ (B * S, D, R) -> (B * S, N, R)
        numerator = torch.bmm(x.transpose(1, 2), bases)
        # (B * S, N, R) @ [(B * S, D, R)^T @ (B * S, D, R)] -> (B * S, N, R)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        # Multiplicative Update
        coef = coef * numerator / (denominator + 1e-6)

        # (B * S, D, N) @ (B * S, N, R) -> (B * S, D, R)
        numerator = torch.bmm(x, coef)
        # (B * S, D, R) @ [(B * S, N, R)^T @ (B * S, N, R)] -> (B * S, D, R)
        denominator = bases.bmm(coef.transpose(1, 2).bmm(coef))
        # Multiplicative Update
        bases = bases * numerator / (denominator + 1e-6)

        return bases, coef

    def compute_coef(self, x, bases, coef):
        # (B * S, D, N)^T @ (B * S, D, R) -> (B * S, N, R)
        numerator = torch.bmm(x.transpose(1, 2), bases)
        # (B * S, N, R) @ (B * S, D, R)^T @ (B * S, D, R) -> (B * S, N, R)
        denominator = coef.bmm(bases.transpose(1, 2).bmm(bases))
        # multiplication update
        coef = coef * numerator / (denominator + 1e-6)

        return coef


class VQ(_MatrixDecompositionBase):
    def __init__(self, device, md_config, debug=False, dim="3D"):
        super().__init__(device, md_config, debug=debug, dim=dim)
        self.device = device

    def _build_bases(self, B, S, D, R):
        # bases = torch.randn((B * S, D, R)).to(self.device)
        bases = torch.ones((B * S, D, R)).to(self.device)
        bases = F.normalize(bases, dim=1)
        return bases

    @torch.no_grad()
    def local_step(self, x, bases, _):
        # (B * S, D, N), normalize x along D (for cosine similarity)
        std_x = F.normalize(x, dim=1)

        # (B * S, D, R), normalize bases along D (for cosine similarity)
        std_bases = F.normalize(bases, dim=1, eps=1e-6)

        # (B * S, D, N)^T @ (B * S, D, R) -> (B * S, N, R)
        coef = torch.bmm(std_x.transpose(1, 2), std_bases)

        # softmax along R
        coef = F.softmax(self.inv_t * coef, dim=-1)

        # normalize along N
        coef = coef / (1e-6 + coef.sum(dim=1, keepdim=True))

        # (B * S, D, N) @ (B * S, N, R) -> (B * S, D, R)
        bases = torch.bmm(x, coef)

        return bases, coef


    def compute_coef(self, x, bases, _):
        with torch.no_grad():
            # (B * S, D, N) -> (B * S, 1, N)
            x_norm = x.norm(dim=1, keepdim=True)

        # (B * S, D, N) / (B * S, 1, N) -> (B * S, D, N)
        std_x = x / (1e-6 + x_norm)

        # (B * S, D, R), normalize bases along D (for cosine similarity)
        std_bases = F.normalize(bases, dim=1, eps=1e-6)

        # (B * S, N, D)^T @ (B * S, D, R) -> (B * S, N, R)
        coef = torch.bmm(std_x.transpose(1, 2), std_bases)

        # softmax along R
        coef = F.softmax(self.inv_t * coef, dim=-1)

        return coef


class ConvBNReLU(nn.Module):
    @classmethod
    def _same_paddings(cls, kernel_size, dim):
        if dim == "3D":
            if kernel_size == (1, 1, 1):
                return (0, 0, 0)
            elif kernel_size == (3, 3, 3):
                return (1, 1, 1)
        elif dim == "2D" or dim == "2D_TSM":
            if kernel_size == (1, 1):
                return (0, 0)
            elif kernel_size == (3, 3):
                return (1, 1)
        else:
            if kernel_size == 1:
                return 0
            elif kernel_size == 3:
                return 1

    def __init__(self, in_c, out_c, dim,
                 kernel_size=1, stride=1, padding='same',
                 dilation=1, groups=1, act='relu', apply_bn=False, apply_act=True):
        super().__init__()

        self.apply_bn = apply_bn
        self.apply_act = apply_act
        self.dim = dim
        if dilation == 1:
            if self.dim == "3D":
                dilation = (1, 1, 1)
            elif self.dim == "2D" or dim == "2D_TSM":
                dilation = (1, 1)
            else:
                dilation = 1

        if kernel_size == 1:
            if self.dim == "3D":
                kernel_size = (1, 1, 1)
            elif self.dim == "2D" or dim == "2D_TSM":
                kernel_size = (1, 1)
            else:
                kernel_size = 1

        if stride == 1:
            if self.dim == "3D":
                stride = (1, 1, 1)
            elif self.dim == "2D" or dim == "2D_TSM":
                stride = (1, 1)
            else:
                stride = 1

        if padding == 'same':
            padding = self._same_paddings(kernel_size, dim)

        if self.dim == "3D":
            self.conv = nn.Conv3d(in_c, out_c,
                                  kernel_size=kernel_size, stride=stride,
                                  padding=padding, dilation=dilation,
                                  groups=groups,
                                  bias=False)
        elif self.dim == "2D" or dim == "2D_TSM":
            self.conv = nn.Conv2d(in_c, out_c,
                                  kernel_size=kernel_size, stride=stride,
                                  padding=padding, dilation=dilation,
                                  groups=groups,
                                  bias=False)
        else:
            self.conv = nn.Conv1d(in_c, out_c,
                                  kernel_size=kernel_size, stride=stride,
                                  padding=padding, dilation=dilation,
                                  groups=groups,
                                  bias=False)

        if act == "sigmoid":
            self.act = nn.Sigmoid()
        else:
            self.act = nn.ReLU(inplace=True)

        if self.apply_bn:
            if self.dim == "3D":
                self.bn = nn.InstanceNorm3d(out_c)
            elif self.dim == "2D" or dim == "2D_TSM":
                self.bn = nn.InstanceNorm2d(out_c)
            else:
                self.bn = nn.InstanceNorm1d(out_c)

    def forward(self, x):
        x = self.conv(x)
        if self.apply_act:
            x = self.act(x)
        if self.apply_bn:
            x = self.bn(x)
        return x


class FeaturesFactorizationModule(nn.Module):
    def __init__(self, inC, device, md_config, dim="3D", debug=False):
        super().__init__()

        self.device = device
        self.dim = dim
        md_type = md_config["MD_TYPE"]
        align_C = md_config["align_channels"]  # inC // 2  # // 2 #// 8

        if self.dim == "3D":
            if "nmf" in md_type.lower():
                self.pre_conv_block = nn.Sequential(
                    nn.Conv3d(inC, align_C, (1, 1, 1)), 
                    nn.ReLU(inplace=True))
            else:
                self.pre_conv_block = nn.Conv3d(inC, align_C, (1, 1, 1))
        elif self.dim == "2D" or self.dim == "2D_TSM":
            if "nmf" in md_type.lower():
                self.pre_conv_block = nn.Sequential(
                    nn.Conv2d(inC, align_C, (1, 1)),
                    nn.ReLU(inplace=True)
                    )
            else:
                self.pre_conv_block = nn.Conv2d(inC, align_C, (1, 1))
        elif self.dim == "1D":
            if "nmf" in md_type.lower():
                self.pre_conv_block = nn.Sequential(
                    nn.Conv1d(inC, align_C, 1),
                    nn.ReLU(inplace=True)
                    )
            else:
                self.pre_conv_block = nn.Conv1d(inC, align_C, 1)
        else:
            print("Dimension not supported")

        if "nmf" in md_type.lower():
            self.md_block = NMF(self.device, md_config, dim=self.dim, debug=debug)
        elif "vq" in md_type.lower():
            self.md_block = VQ(self.device, md_config, dim=self.dim, debug=debug)
        else:
            print("Unknown type specified for MD_TYPE:", md_type)
            exit()

        if self.dim == "3D":
            if "nmf" in md_type.lower():
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1),
                    nn.Conv3d(align_C, inC, 1, bias=False)
                    )
            else:
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1, apply_act=False), 
                    nn.Conv3d(align_C, inC, 1, bias=False)
                    )
        elif self.dim == "2D" or self.dim == "2D_TSM":
            if "nmf" in md_type.lower():
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1), 
                    nn.Conv2d(align_C, inC, 1, bias=False)
                    )
            else:
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1, apply_act=False),
                    nn.Conv2d(align_C, inC, 1, bias=False)
                    )
        else:
            if "nmf" in md_type.lower():
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1),
                    nn.Conv1d(align_C, inC, 1, bias=False)
                    )
            else:
                self.post_conv_block = nn.Sequential(
                    ConvBNReLU(align_C, align_C, dim=self.dim, kernel_size=1, apply_act=False),
                    nn.Conv1d(align_C, inC, 1, bias=False)
                    )

        self._init_weight()


    def _init_weight(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                N = m.kernel_size[0] * m.kernel_size[1] * m.kernel_size[2] * m.out_channels
                m.weight.data.normal_(0, np.sqrt(2. / N))
            elif isinstance(m, nn.Conv2d):
                N = m.kernel_size[0] * m.kernel_size[1] * m.out_channels
                m.weight.data.normal_(0, np.sqrt(2. / N))
            elif isinstance(m, nn.Conv1d):
                N = m.kernel_size[0] * m.out_channels
                m.weight.data.normal_(0, np.sqrt(2. / N))
            elif isinstance(m, _BatchNorm):
                m.weight.data.fill_(1)
                if m.bias is not None:
                    m.bias.data.zero_()

    def forward(self, x):
        x = self.pre_conv_block(x)
        att = self.md_block(x)
        dist = torch.dist(x, att)
        att = self.post_conv_block(att)

        return att, dist

    def online_update(self, bases):
        if hasattr(self.md_block, 'online_update'):
            self.md_block.online_update(bases)



In [ ]:
# === inlined from neural_methods/model/FactorizePhys/FactorizePhys.py ===
"""
FactorizePhys: Matrix Factorization for Multidimensional Attention in Remote Physiological Sensing
NeurIPS 2024
Jitesh Joshi, Sos S. Agaian, and Youngjun Cho
"""

import torch
import torch.nn as nn


nf = [8, 12, 16]

model_config = {
    "MD_FSAM": True,
    "MD_TYPE": "NMF",
    "MD_TRANSFORM": "T_KAB",
    "MD_R": 1,
    "MD_S": 1,
    "MD_STEPS": 4,
    "MD_INFERENCE": False,
    "MD_RESIDUAL": False,
    "INV_T": 1,
    "ETA": 0.9,
    "RAND_INIT": True,
    "in_channels": 3,
    "data_channels": 4,
    "align_channels": nf[2] // 2,
    "height": 72,
    "weight": 72,
    "batch_size": 4,
    "frames": 160,
    "debug": False,
    "assess_latency": False,
    "num_trials": 20,
    "visualize": False,
    "ckpt_path": "",
    "data_path": "",
    "label_path": ""
}


class ConvBlock3D(nn.Module):
    def __init__(self, in_channel, out_channel, kernel_size, stride, padding):
        super(ConvBlock3D, self).__init__()
        self.conv_block_3d = nn.Sequential(
            nn.Conv3d(in_channel, out_channel, kernel_size, stride, padding=padding, bias=False),
            nn.Tanh(),
            nn.InstanceNorm3d(out_channel),
        )

    def forward(self, x):
        return self.conv_block_3d(x)


class rPPG_FeatureExtractor(nn.Module):
    def __init__(self, inCh, dropout_rate=0.1, debug=False):
        super(rPPG_FeatureExtractor, self).__init__()
        # inCh, out_channel, kernel_size, stride, padding

        self.debug = debug
        #                                                        Input: #B, inCh, 160, 72, 72
        self.FeatureExtractor = nn.Sequential(
            ConvBlock3D(inCh, nf[0], [3, 3, 3], [1, 1, 1], [1, 1, 1]),  #B, nf[0], 160, 72, 72
            ConvBlock3D(nf[0], nf[1], [3, 3, 3], [1, 2, 2], [1, 0, 0]), #B, nf[1], 160, 35, 35
            ConvBlock3D(nf[1], nf[1], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[1], 160, 33, 33
            nn.Dropout3d(p=dropout_rate),

            ConvBlock3D(nf[1], nf[1], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[1], 160, 31, 31
            ConvBlock3D(nf[1], nf[2], [3, 3, 3], [1, 2, 2], [1, 0, 0]), #B, nf[2], 160, 15, 15
            ConvBlock3D(nf[2], nf[2], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[2], 160, 13, 13
            nn.Dropout3d(p=dropout_rate),
        )

    def forward(self, x):
        voxel_embeddings = self.FeatureExtractor(x)
        if self.debug:
            print("rPPG Feature Extractor")
            print("     voxel_embeddings.shape", voxel_embeddings.shape)
        return voxel_embeddings


class BVP_Head(nn.Module):
    def __init__(self, md_config, device, dropout_rate=0.1, debug=False):
        super(BVP_Head, self).__init__()
        self.debug = debug

        self.use_fsam = md_config["MD_FSAM"]
        self.md_type = md_config["MD_TYPE"]
        self.md_infer = md_config["MD_INFERENCE"]
        self.md_res = md_config["MD_RESIDUAL"]

        self.conv_block = nn.Sequential(
            ConvBlock3D(nf[2], nf[2], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[2], 160, 11, 11
            ConvBlock3D(nf[2], nf[2], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[2], 160, 9, 9
            ConvBlock3D(nf[2], nf[2], [3, 3, 3], [1, 1, 1], [1, 0, 0]), #B, nf[2], 160, 7, 7
            nn.Dropout3d(p=dropout_rate),
        )

        if self.use_fsam:
            inC = nf[2]
            self.fsam = FeaturesFactorizationModule(inC, device, md_config, dim="3D", debug=debug)
            self.fsam_norm = nn.InstanceNorm3d(inC)
            self.bias1 = nn.Parameter(torch.tensor(1.0), requires_grad=True).to(device)
        else:
            inC = nf[2]

        self.final_layer = nn.Sequential(
            ConvBlock3D(inC, nf[1], [3, 3, 3], [1, 1, 1], [1, 0, 0]),                         #B, nf[1], 160, 5, 5
            ConvBlock3D(nf[1], nf[0], [3, 3, 3], [1, 1, 1], [1, 0, 0]),                       #B, nf[0], 160, 3, 3
            nn.Conv3d(nf[0], 1, (3, 3, 3), stride=(1, 1, 1), padding=(1, 0, 0), bias=False),  #B, 1, 160, 1, 1
        )


    def forward(self, voxel_embeddings, batch, length):

        if self.debug:
            print("BVP Head")
            print("     voxel_embeddings.shape", voxel_embeddings.shape)

        voxel_embeddings = self.conv_block(voxel_embeddings)

        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            if "NMF" in self.md_type:
                att_mask, appx_error = self.fsam(voxel_embeddings - voxel_embeddings.min()) # to make it positive (>= 0)
            else:
                att_mask, appx_error = self.fsam(voxel_embeddings)

            if self.debug:
                print("att_mask.shape", att_mask.shape)

            # # directly use att_mask   ---> difficult to converge without Residual connection. Needs high rank
            # factorized_embeddings = self.fsam_norm(att_mask)

            # # Residual connection: 
            # factorized_embeddings = voxel_embeddings + self.fsam_norm(att_mask)

            if self.md_res:
                # Multiplication with Residual connection
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1, att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)
                factorized_embeddings = voxel_embeddings + factorized_embeddings
            else:
                # Multiplication
                x = torch.mul(voxel_embeddings - voxel_embeddings.min() + self.bias1, att_mask - att_mask.min() + self.bias1)
                factorized_embeddings = self.fsam_norm(x)            

            # # Concatenate
            # factorized_embeddings = torch.cat([voxel_embeddings, self.fsam_norm(x)], dim=1)

            x = self.final_layer(factorized_embeddings)
        
        else:
            x = self.final_layer(voxel_embeddings)

        rPPG = x.view(-1, length)

        if self.debug:
            print("     rPPG.shape", rPPG.shape)
        
        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            return rPPG, factorized_embeddings, appx_error
        else:
            return rPPG



class FactorizePhys(nn.Module):
    def __init__(self, frames, md_config, in_channels=3, dropout=0.1, device=torch.device("cpu"), debug=False):
        super(FactorizePhys, self).__init__()
        self.debug = debug

        self.in_channels = in_channels
        if self.in_channels == 1 or self.in_channels == 3:
            self.norm = nn.InstanceNorm3d(self.in_channels)
        elif self.in_channels == 4:
            self.rgb_norm = nn.InstanceNorm3d(3)
            self.thermal_norm = nn.InstanceNorm3d(1)
        else:
            print("Unsupported input channels")
        
        self.use_fsam = md_config["MD_FSAM"]
        self.md_infer = md_config["MD_INFERENCE"]

        for key in model_config:
            if key not in md_config:
                md_config[key] = model_config[key]

        if self.debug:
            print("nf:", nf)

        self.rppg_feature_extractor = rPPG_FeatureExtractor(self.in_channels, dropout_rate=dropout, debug=debug)

        self.rppg_head = BVP_Head(md_config, device=device, dropout_rate=dropout, debug=debug)

        
    def forward(self, x): # [batch, Features=3, Temp=frames, Width=32, Height=32]
        
        [batch, channel, length, width, height] = x.shape
        
        # if self.in_channels == 1:
        #     x = x[:, :, :-1, :, :]
        # else:
        #     x = torch.diff(x, dim=2)
        
        x = torch.diff(x, dim=2)

        if self.debug:
            print("Input.shape", x.shape)

        if self.in_channels == 1:
            x = self.norm(x[:, -1:, :, :, :])
        elif self.in_channels == 3:
            x = self.norm(x[:, :3, :, :, :])
        elif self.in_channels == 4:
            rgb_x = self.rgb_norm(x[:, :3, :, :, :])
            thermal_x = self.thermal_norm(x[:, -1:, :, :, :])
            x = torch.concat([rgb_x, thermal_x], dim = 1)
        else:
            try:
                print("Specified input channels:", self.in_channels)
                print("Data channels", channel)
                assert self.in_channels <= channel
            except:
                print("Incorrectly preprocessed data provided as input. Number of channels exceed the specified or default channels")
                print("Default or specified channels:", self.in_channels)
                print("Data channels [B, C, N, W, H]", x.shape)
                print("Exiting")
                exit()

        if self.debug:
            print("Diff Normalized shape", x.shape)

        voxel_embeddings = self.rppg_feature_extractor(x)
        
        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            rPPG, factorized_embeddings, appx_error = self.rppg_head(voxel_embeddings, batch, length-1)
        else:
            rPPG = self.rppg_head(voxel_embeddings, batch, length-1)

        # if self.debug:
        #     print("rppg_feats.shape", rppg_feats.shape)

        # rPPG = rppg_feats.view(-1, length-1)

        if self.debug:
            print("rPPG.shape", rPPG.shape)

        if (self.md_infer or self.training or self.debug) and self.use_fsam:
            return rPPG, voxel_embeddings, factorized_embeddings, appx_error
        else:
            return rPPG, voxel_embeddings

In [ ]:
# === inlined from neural_methods/loss/NegPearsonLoss.py ===
from __future__ import print_function, division
import torch
import matplotlib.pyplot as plt
import argparse, os
import pandas as pd
import numpy as np
import random
import math
from torchvision import transforms
from torch import nn


class Neg_Pearson(nn.Module):
    def __init__(self):
        super(Neg_Pearson, self).__init__()
        return

    def forward(self, preds, labels):
        cos = nn.CosineSimilarity(dim=0, eps=1e-6)
        pearson = cos(preds - preds.mean(dim=0, keepdim=True), labels - labels.mean(dim=0, keepdim=True))
        return torch.mean(1 - pearson)




## Training utilities

Seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.


In [ ]:
# === Training utilities (shared across notebooks) ===
# seed, train/val split, HR-MAE, best-checkpoint saver, early-stopping, CSV logger.
import os
import random as _random
import csv as _csv
import time as _time
import numpy as _np
import torch as _torch
from scipy.signal import periodogram as _periodogram


def set_seed(seed: int = 42):
    """Set seeds for reproducibility across random, numpy, torch (CPU + CUDA)."""
    _random.seed(seed)
    _np.random.seed(seed)
    _torch.manual_seed(seed)
    _torch.cuda.manual_seed_all(seed)
    _torch.backends.cudnn.deterministic = True
    _torch.backends.cudnn.benchmark = False


def train_val_split(dataset, val_ratio: float = 0.2, seed: int = 42):
    """Random split returning (train_subset, val_subset) with a seeded generator."""
    n_total = len(dataset)
    n_val = max(1, int(n_total * val_ratio))
    n_train = n_total - n_val
    g = _torch.Generator().manual_seed(seed)
    return _torch.utils.data.random_split(dataset, [n_train, n_val], generator=g)


def compute_hr_fft(signal_1d, fps: int = 30, lo_hz: float = 0.6, hi_hz: float = 3.3) -> float:
    """Peak-frequency HR (bpm) of a 1-D signal via periodogram, restricted to a band."""
    sig = signal_1d.detach().cpu().numpy() if _torch.is_tensor(signal_1d) else _np.asarray(signal_1d)
    sig = sig.astype(_np.float64).ravel()
    if sig.size < 8 or sig.std() < 1e-8:
        return 0.0
    sig = sig - sig.mean()
    freqs, psd = _periodogram(sig, fs=fps)
    band = (freqs >= lo_hz) & (freqs <= hi_hz)
    if not band.any():
        return 0.0
    return float(freqs[band][psd[band].argmax()] * 60.0)


def compute_hr_mae_batch(preds, labels, fps: int = 30) -> float:
    """Mean absolute HR error (bpm) over a batch.  preds/labels: (N, T) or (T,)."""
    if preds.dim() == 1:
        preds = preds.unsqueeze(0)
    if labels.dim() == 1:
        labels = labels.unsqueeze(0)
    errs = []
    for i in range(preds.shape[0]):
        hr_p = compute_hr_fft(preds[i], fps)
        hr_l = compute_hr_fft(labels[i], fps)
        errs.append(abs(hr_p - hr_l))
    return float(_np.mean(errs)) if errs else 0.0


class BestCheckpointSaver:
    """Save the model's state_dict whenever a tracked metric improves."""
    def __init__(self, path: str, mode: str = "min"):
        assert mode in ("min", "max")
        self.path = path
        self.mode = mode
        self.best = float("inf") if mode == "min" else -float("inf")
        os.makedirs(os.path.dirname(os.path.abspath(path)) or ".", exist_ok=True)

    def step(self, model, metric: float) -> bool:
        improved = (metric < self.best) if self.mode == "min" else (metric > self.best)
        if improved and not (metric != metric):  # reject NaN
            self.best = metric
            _torch.save(model.state_dict(), self.path)
            return True
        return False


class EarlyStopping:
    """Stop training when the tracked metric stops improving for `patience` epochs."""
    def __init__(self, patience: int = 5, mode: str = "min", min_delta: float = 0.0):
        assert mode in ("min", "max")
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.best = float("inf") if mode == "min" else -float("inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metric: float) -> bool:
        improved = (
            (self.mode == "min" and metric < self.best - self.min_delta)
            or (self.mode == "max" and metric > self.best + self.min_delta)
        )
        if improved:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        return self.should_stop


class MetricLogger:
    """Append per-epoch metrics to a CSV.  Creates the file with headers on init."""
    def __init__(self, csv_path: str, fieldnames=None):
        self.csv_path = csv_path
        self.fieldnames = list(fieldnames) if fieldnames else [
            "epoch", "train_loss", "val_loss", "val_hr_mae", "lr", "time_sec"
        ]
        os.makedirs(os.path.dirname(os.path.abspath(csv_path)) or ".", exist_ok=True)
        with open(csv_path, "w", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writeheader()

    def log(self, **row):
        with open(self.csv_path, "a", newline="") as f:
            _csv.DictWriter(f, fieldnames=self.fieldnames).writerow(
                {k: row.get(k, "") for k in self.fieldnames}
            )


# Seed the run for reproducibility
set_seed(42)
print("Training utils ready  |  seed=42  |  HR-MAE band: 0.6-3.3 Hz (36-198 bpm)")


In [3]:
# ----- Configs -----
PREPROCESSED_PATH = os.path.join(REPO_ROOT, "preprocessed_data/Normal/groupF")
OUTPUT_DIR = os.path.join(REPO_ROOT, "final_model_release")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNK_LENGTH = 160
BATCH_SIZE = 4
EPOCHS = 30
LR = 3e-4          # was 1e-3 -> too aggressive vs other groups; 3e-4 is a safer max
PATIENCE = 5
SEED = 42
VAL_RATIO = 0.2
VIDEO_FPS = 30

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Device: cuda:0


In [4]:
# ----- Dataset, train/val split, DataLoaders -----
class GroupFDataset(Dataset):
    def __init__(self, preprocessed_dir):
        self.inputs = sorted(glob.glob(os.path.join(preprocessed_dir, "*", "*_input*.npy")))
        self.labels = [f.replace("input", "label") for f in self.inputs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, index):
        data  = np.float32(np.load(self.inputs[index]))   # (T, H, W, 3)
        label = np.float32(np.load(self.labels[index]))   # (T,)
        data = np.transpose(data, (3, 0, 1, 2))            # -> (3, T, H, W)
        fname = os.path.basename(self.inputs[index])
        try:
            split_idx  = fname.index("_")
            subject_id = fname[:split_idx]
            chunk_id   = fname[split_idx + 6:].split(".")[0]
        except ValueError:
            subject_id, chunk_id = "unknown", "0"
        return data, label, subject_id, chunk_id


dataset = GroupFDataset(PREPROCESSED_PATH)
print(f"Total clips found: {len(dataset)}")
train_ds, val_ds = train_val_split(dataset, val_ratio=VAL_RATIO, seed=SEED)
g = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, generator=g)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)} ({len(train_loader)} batches)  |  Val: {len(val_ds)} ({len(val_loader)} batches)")


Total clips found: 160
DataLoader configured with 40 batches.


## 1. Train iBVPNet

In [ ]:
def train_ibvpnet():
    print("\n--- Training iBVPNet ---")
    save_path = os.path.join(OUTPUT_DIR, "GroupF_iBVPNet.pth")
    log_path  = os.path.join(REPO_ROOT, "results/Normal/groupF/train_logs/iBVPNet.csv")
    model = iBVPNet(frames=CHUNK_LENGTH, in_channels=3).to(DEVICE)
    criterion = Neg_Pearson()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
    )
    saver   = BestCheckpointSaver(save_path, mode="min")
    stopper = EarlyStopping(patience=PATIENCE, mode="min")
    logger  = MetricLogger(log_path)

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        train_loss_sum, n_train = 0.0, 0
        tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
        for batch in tbar:
            data, labels = batch[0].to(DEVICE, non_blocking=True), batch[1].to(DEVICE, non_blocking=True)
            data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
            pred = model(data_padded)
            pred_n   = (pred   - pred.mean())   / (pred.std()   + 1e-8)
            labels_n = (labels - labels.mean()) / (labels.std() + 1e-8)
            loss = criterion(pred_n, labels_n)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss_sum += loss.item()
            n_train += 1
            tbar.set_postfix(loss=f"{loss.item():.4f}")
        train_loss = train_loss_sum / max(1, n_train)

        model.eval()
        val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
                data, labels = batch[0].to(DEVICE, non_blocking=True), batch[1].to(DEVICE, non_blocking=True)
                data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
                pred = model(data_padded)
                pred_n   = (pred   - pred.mean())   / (pred.std()   + 1e-8)
                labels_n = (labels - labels.mean()) / (labels.std() + 1e-8)
                val_loss_sum += criterion(pred_n, labels_n).item()
                val_hr_mae_sum += compute_hr_mae_batch(pred, labels, fps=VIDEO_FPS) * pred.shape[0]
                n_val += pred.shape[0]
        val_loss   = val_loss_sum / max(1, len(val_loader))
        val_hr_mae = val_hr_mae_sum / max(1, n_val)
        elapsed = time.time() - t0
        cur_lr = optimizer.param_groups[0]["lr"]
        improved = saver.step(model, val_hr_mae)
        marker = " <- best" if improved else ""
        print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_HR_MAE={val_hr_mae:.2f} bpm  lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
        logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
                   val_hr_mae=val_hr_mae, lr=cur_lr, time_sec=elapsed)
        if stopper.step(val_hr_mae):
            print(f"Early stopping at epoch {epoch+1}")
            break
    print(f"Best val HR-MAE: {saver.best:.2f} bpm  ->  {save_path}")


In [ ]:
# train_ibvpnet()


## 2. Train FactorizePhys

In [5]:
def train_factorizephys():
    print("\n--- Training FactorizePhys ---")
    save_path = os.path.join(OUTPUT_DIR, "GroupF_FactorizePhys.pth")
    log_path  = os.path.join(REPO_ROOT, "results/Normal/groupF/train_logs/FactorizePhys.csv")
    MD_CONFIG = {
        "FRAME_NUM": CHUNK_LENGTH, "MD_FSAM": True, "MD_TYPE": "NMF",
        "MD_TRANSFORM": "T_KAB", "MD_R": 1, "MD_S": 1, "MD_STEPS": 3,
        "MD_INFERENCE": False, "MD_RESIDUAL": True,
    }
    model = FactorizePhys(
        frames=CHUNK_LENGTH, md_config=MD_CONFIG, in_channels=3,
        dropout=0.1, device=torch.device(DEVICE),
    ).to(DEVICE)
    criterion = Neg_Pearson()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_loader)
    )
    saver   = BestCheckpointSaver(save_path, mode="min")
    stopper = EarlyStopping(patience=PATIENCE, mode="min")
    logger  = MetricLogger(log_path)

    for epoch in range(EPOCHS):
        t0 = time.time()
        model.train()
        train_loss_sum, n_train = 0.0, 0
        tbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [train]", ncols=90)
        for batch in tbar:
            data, labels = batch[0].to(DEVICE, non_blocking=True), batch[1].to(DEVICE, non_blocking=True)
            if labels.dim() > 2:
                labels = labels[..., 0]
            data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
            pred, _, _, appx_err = model(data_padded)
            pred_n   = (pred   - pred.mean())   / (pred.std()   + 1e-8)
            labels_n = (labels - labels.mean()) / (labels.std() + 1e-8)
            loss = criterion(pred_n, labels_n)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            train_loss_sum += loss.item()
            n_train += 1
            tbar.set_postfix(loss=f"{loss.item():.4f}", appx=f"{appx_err.item():.3f}")
        train_loss = train_loss_sum / max(1, n_train)

        model.eval()
        val_loss_sum, val_hr_mae_sum, n_val = 0.0, 0.0, 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [val]  ", ncols=90):
                data, labels = batch[0].to(DEVICE, non_blocking=True), batch[1].to(DEVICE, non_blocking=True)
                if labels.dim() > 2:
                    labels = labels[..., 0]
                data_padded = torch.cat([data, data[:, :, -1:].clone()], dim=2)
                pred, _, _, _ = model(data_padded)
                pred_n   = (pred   - pred.mean())   / (pred.std()   + 1e-8)
                labels_n = (labels - labels.mean()) / (labels.std() + 1e-8)
                val_loss_sum += criterion(pred_n, labels_n).item()
                val_hr_mae_sum += compute_hr_mae_batch(pred, labels, fps=VIDEO_FPS) * pred.shape[0]
                n_val += pred.shape[0]
        val_loss   = val_loss_sum / max(1, len(val_loader))
        val_hr_mae = val_hr_mae_sum / max(1, n_val)
        elapsed = time.time() - t0
        cur_lr = optimizer.param_groups[0]["lr"]
        improved = saver.step(model, val_hr_mae)
        marker = " <- best" if improved else ""
        print(f"Epoch {epoch+1:2d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
              f"val_HR_MAE={val_hr_mae:.2f} bpm  lr={cur_lr:.2e}  ({elapsed:.1f}s){marker}")
        logger.log(epoch=epoch+1, train_loss=train_loss, val_loss=val_loss,
                   val_hr_mae=val_hr_mae, lr=cur_lr, time_sec=elapsed)
        if stopper.step(val_hr_mae):
            print(f"Early stopping at epoch {epoch+1}")
            break
    print(f"Best val HR-MAE: {saver.best:.2f} bpm  ->  {save_path}")


In [6]:
train_factorizephys()



--- Training FactorizePhys ---


Epoch 1/10: 100%|██████████| 40/40 [00:04<00:00,  8.07it/s, loss=0.9538, appx_err=881.2657]


Epoch 1 Avg Loss: 0.9266


Epoch 2/10: 100%|██████████| 40/40 [00:01<00:00, 22.14it/s, loss=0.3513, appx_err=719.7794]


Epoch 2 Avg Loss: 0.4671


...

Epoch 8 Avg Loss: 0.2699


Epoch 9/10: 100%|██████████| 40/40 [00:01<00:00, 22.72it/s, loss=0.4219, appx_err=792.2517]


Epoch 9 Avg Loss: 0.2435


Epoch 10/10: 100%|██████████| 40/40 [00:01<00:00, 21.36it/s, loss=0.3717, appx_err=827.0134]

Epoch 10 Avg Loss: 0.2520
Saved FactorizePhys to /home/iec/MinhHieu/Non-Invasive/rPPG/final_model_release/GroupF_FactorizePhys.pth
